### Construct a model (HH neurons with STDP plasticity) TODO: RENAME IT

https://brian2.readthedocs.io/en/stable/user/models.html

In [ ]:
import brian2 as b2
import numpy as np

from neuroseq.b2_model import B2Model

In [ ]:
tfinal = 1000 * b2.ms

b2.seed(42)

nneurons = 10

# Parameters
area = 20000 * b2.umetre**2
Cm = (1 * b2.ufarad * b2.cm**-2) * area
gl = (5e-5 * b2.siemens * b2.cm**-2) * area

El = -60 * b2.mV
EK = -90 * b2.mV
ENa = 50 * b2.mV
g_na = (100 * b2.msiemens * b2.cm**-2) * area
g_kd = (30 * b2.msiemens * b2.cm**-2) * area


VT = -40 * b2.mV
# Time constants
taue = 5 * b2.ms
taui = 10 * b2.ms
# Reversal potentials
Eeopto = 0 * b2.mV
Eiopto = -80 * b2.mV
g_exc = (2e-3 * b2.siemens * b2.cm**-2) * area
g_inh = (3e-3 * b2.siemens * b2.cm**-2) * area

opto_units = 1 * b2.mwatt / b2.um2

i_stim = np.zeros((int(tfinal / b2.ms), nneurons), dtype="float")
i_stim[:, :] = 0
i_stim[200:300, 0] = 1
I_external = b2.TimedArray(i_stim * b2.pA, dt=1 * b2.ms)

exc_stim = np.zeros((int(tfinal / b2.ms), nneurons), dtype="float")
exc_stim[:, :] = 0
exc_stim[0:200, 0] = 1
Exc_stimulation = b2.TimedArray(exc_stim * opto_units, dt=1 * b2.ms)

inh_stim = np.zeros((int(tfinal / b2.ms), nneurons), dtype="float")
inh_stim[:, :] = 0
inh_stim[300:900, 0] = 1
Inh_stimulation = b2.TimedArray(inh_stim * opto_units, dt=1 * b2.ms)

# The model
eqs = b2.Equations("""
dv/dt = (geopto*(Eeopto-v)+giopto*(Eiopto-v)+gl*(El-v)-
         g_na*(m*m*m)*h*(v-ENa)-
         g_kd*(n*n*n*n)*(v-EK)+Iext)/Cm: volt
Iext = I_external(t,i) : amp
geopto = g_exc * Exc_stimulation(t, i)/opto_units: siemens
giopto = g_inh * Inh_stimulation(t, i)/opto_units: siemens
dm/dt = alpha_m*(1-m)-beta_m*m : 1
dn/dt = alpha_n*(1-n)-beta_n*n : 1
dh/dt = alpha_h*(1-h)-beta_h*h : 1
alpha_m = 0.32*(mV**-1)*4*mV/exprel((13*mV-v+VT)/(4*mV))/ms : Hz
beta_m = 0.28*(mV**-1)*5*mV/exprel((v-VT-40*mV)/(5*mV))/ms : Hz
alpha_h = 0.128*exp((17*mV-v+VT)/(18*mV))/ms : Hz
beta_h = 4./(1+exp((40*mV-v+VT)/(5*mV)))/ms : Hz
alpha_n = 0.032*(mV**-1)*5*mV/exprel((15*mV-v+VT)/(5*mV))/ms : Hz
beta_n = .5*exp((10*mV-v+VT)/(40*mV))/ms : Hz
""")

P = b2.NeuronGroup(
    nneurons,
    model=eqs,
    threshold="v>-20*mV",
    refractory=3 * b2.ms,
    method="exponential_euler",
)
#
# Initialization
P.v = "El + (randn() * 5 - 5)*mV"

# Record a few traces
trace = b2.StateMonitor(P, "v", record=True)
b2.run(1 * b2.second, report="text")
b2.plot(trace.t / b2.ms, trace[0].v / b2.mV)
# plot(trace.t/ms, trace[10].v/mV)
# plot(trace.t/ms, trace[100].v/mV)
b2.xlabel("t (ms)")
b2.ylabel("v (mV)")
b2.show()

In [ ]:
b2.plot(trace.t / b2.ms, trace[1].v / b2.mV)
b2.plot(trace.t / b2.ms, trace[2].v / b2.mV)
b2.plot(trace.t / b2.ms, trace[3].v / b2.mV)
b2.xlabel("t (ms)")
b2.ylabel("v (mV)")
b2.show()

### b2m class implementation

In [ ]:
# b2m class implementation

model = B2Model(seed=42)

tfinal = 1000 * b2.ms

nneurons = 10

model.define_parameter("area", 20000 * b2.umetre**2)
model.define_parameter("Cm", (1 * b2.ufarad * b2.cm**-2) * model.parameters["area"])
model.define_parameter("gl", (5e-5 * b2.siemens * b2.cm**-2) * model.parameters["area"])

model.define_parameter("El", -60 * b2.mV)
model.define_parameter("EK", -90 * b2.mV)
model.define_parameter("ENa", 50 * b2.mV)
model.define_parameter(
    "g_na", (100 * b2.msiemens * b2.cm**-2) * model.parameters["area"]
)
model.define_parameter(
    "g_kd", (30 * b2.msiemens * b2.cm**-2) * model.parameters["area"]
)
model.define_parameter("VT", -40 * b2.mV)
# Time constants
model.define_parameter("taue", 5 * b2.ms)
model.define_parameter("taui", 10 * b2.ms)
# # Reversal potentials
model.define_parameter("Eeopto", 0 * b2.mV)
model.define_parameter("Eiopto", -80 * b2.mV)
model.define_parameter(
    "g_exc", (2e-3 * b2.siemens * b2.cm**-2) * model.parameters["area"]
)
model.define_parameter(
    "g_inh", (3e-3 * b2.siemens * b2.cm**-2) * model.parameters["area"]
)

opto_units = 1 * b2.mwatt / b2.um2

model.define_parameter("opto_units", opto_units)


i_stim = np.zeros((int(tfinal / b2.ms), nneurons), dtype="float")
i_stim[:, :] = 0
i_stim[200:300, 0] = 1
I_external = b2.TimedArray(i_stim * b2.pA, dt=1 * b2.ms)

exc_stim = np.zeros((int(tfinal / b2.ms), nneurons), dtype="float")
exc_stim[:, :] = 0
exc_stim[0:200, 0] = 1
Exc_stimulation = b2.TimedArray(exc_stim * opto_units, dt=1 * b2.ms)

inh_stim = np.zeros((int(tfinal / b2.ms), nneurons), dtype="float")
inh_stim[:, :] = 0
inh_stim[300:900, 0] = 1
Inh_stimulation = b2.TimedArray(inh_stim * opto_units, dt=1 * b2.ms)

# +sigma*sqrt(2/tau)*xi

model.define_parameter("I_external", I_external)
model.define_parameter("Exc_stimulation", Exc_stimulation)
model.define_parameter("Inh_stimulation", Inh_stimulation)


eqs = """
dv/dt = (geopto*(Eeopto-v)+giopto*(Eiopto-v)+gl*(El-v)-
         g_na*(m*m*m)*h*(v-ENa)-
         g_kd*(n*n*n*n)*(v-EK)+Iext)/Cm : volt
Iext = I_external(t,i) : amp
geopto = g_exc * Exc_stimulation(t, i)/opto_units: siemens
giopto = g_inh * Inh_stimulation(t, i)/opto_units: siemens
dm/dt = alpha_m*(1-m)-beta_m*m : 1
dn/dt = alpha_n*(1-n)-beta_n*n : 1
dh/dt = alpha_h*(1-h)-beta_h*h : 1
dge/dt = -ge*(1./taue) : siemens
dgi/dt = -gi*(1./taui) : siemens
alpha_m = 0.32*(mV**-1)*4*mV/exprel((13*mV-v+VT)/(4*mV))/ms : Hz
beta_m = 0.28*(mV**-1)*5*mV/exprel((v-VT-40*mV)/(5*mV))/ms : Hz
alpha_h = 0.128*exp((17*mV-v+VT)/(18*mV))/ms : Hz
beta_h = 4./(1+exp((40*mV-v+VT)/(5*mV)))/ms : Hz
alpha_n = 0.032*(mV**-1)*5*mV/exprel((15*mV-v+VT)/(5*mV))/ms : Hz
beta_n = .5*exp((10*mV-v+VT)/(40*mV))/ms : Hz
"""

# neuron group
model.define_neuronnumber("neuron", nneurons)
model.define_solver_method("neuron", "exponential_euler")
model.define_equation("neuron", eqs)
model.define_threshold("neuron", "v>-20*mV")
# model.define_reset("neuron", "v=-65*mV")
model.define_refractory("neuron", 2 * b2.ms)
model.define_dt("neuron", 0.1 * b2.ms)
model._assemble_neurongroup("neuron")

# initial conditions
model.define_init("neuron", "v", "El + (randn() * 5 - 5)*mV")

# monitors
model.define_monitor_source("trace", "neuron")
model.define_monitor_type("trace", "StateMonitor")
model.define_monitor_variable("trace", True)
# model.define_monitor_variable("trace", "Istim")
model._assemble_monitor("trace")

model.define_monitor_source("mon", "neuron")
model.define_monitor_type("mon", "SpikeMonitor")
model._assemble_monitor("mon")

model.assemble_network()

In [ ]:
model.save_model("../src/neuroseq/example_models/example_non-HH_model.nseq")

In [ ]:
model.run_simulation(tfinal)

In [ ]:
trace = model.get_monitor("trace")["object"]
# mon = model.get_monitor("mon")["object"]
b2.plot(trace.t / b2.ms, trace.v[0] / b2.mV, lw=0.8)
# b2.plot(mon.t/b2.ms,mon.i+30,'k|')
# b2.plot(trace.t/b2.ms, trace.v[0]/b2.mV)
# b2.plot(trace.t/b2.ms, trace.v[10]/b2.mV)
# b2.plot(trace.t/b2.ms, trace.v[100]/b2.mV)
b2.xlabel("t (ms)")
b2.ylabel("v (mV)")
b2.show()

In [ ]:
b2.plot(trace.t / b2.ms, trace[1].v / b2.mV)
b2.plot(trace.t / b2.ms, trace[2].v / b2.mV)
b2.plot(trace.t / b2.ms, trace[3].v / b2.mV)
b2.xlabel("t (ms)")
b2.ylabel("v (mV)")
b2.show()